In [ ]:
# list — используйте для упорядоченных последовательностей, по которым итерируетесь
prices = [10.5, 20.0, 15.75]
prices.append(8.99)
print(prices[-1])

In [ ]:
# tuple — используйте для фиксированных записей (координаты, строки БД)
point = (52.52, 13.40)

In [ ]:
# dict — используйте для именованных данных / таблиц поиска
country_gdp = {"Germany": 4.07, "France": 2.78, "Italy": 2.10}
print(country_gdp.get("Spain", 0))

In [ ]:
# set — используйте для проверки членства и дедупликации
categories = {"electronics", "clothing", "food", "clothing"}
print(categories) 

In [1]:
# O(1) vs O(n) — важно при больших объёмах данных
large_set = set(range(1_000_000))
large_list = list(range(1_000_000))

In [ ]:
%%time
10000 in large_set

In [ ]:
%%time
1200 in large_list

In [ ]:
# Хешируемые типы (могут быть ключом dict / элементом set)
{1: "ok"}
{(1, 2): "ok"}       

In [ ]:
# Нехешируемые типы
{[1, 2]: "fail"}   
{{}: "fail"} 

#### `is` vs `==`

In [ ]:
a = [1, 2, 3]
b = [1, 2, 3]
c = a

print(a == b)
print(a is b)
print(a is c)

c.append(4)
print(a)

#### Поверхностное vs глубокое копирование

In [ ]:
import copy

original = {"name": "Alice", "scores": [90, 85, 78]}

# Поверхностное копирование — верхний уровень новый, вложенные объекты общие
shallow = original.copy()
shallow["name"] = "Bob"          # ok — оригинал не затронут
shallow["scores"].append(70)     # БАГ — изменяет список оригинала тоже!
print(original["scores"])        # [90, 85, 78, 70]  ← изменён

# Глубокое копирование — полностью независимый объект
original = {"name": "Alice", "scores": [90, 85, 78]}
deep = copy.deepcopy(original)
deep["scores"].append(70)
print(original["scores"])        # [90, 85, 78]  ← в безопасности

In [ ]:
# ❌ Неверно — все строки указывают на один и тот же список
matrix = [[0] * 3] * 3
matrix[0][0] = 1
print(matrix)  # [[1, 0, 0], [1, 0, 0], [1, 0, 0]] ← не то, что вы хотели

# ✅ Верно — каждая строка является независимым списком
matrix = [[0] * 3 for _ in range(3)]
matrix[0][0] = 1
print(matrix)  # [[1, 0, 0], [0, 0, 0], [0, 0, 0]]

### Модуль `collections`

In [ ]:
from collections import Counter, defaultdict, namedtuple, OrderedDict

In [ ]:
from collections import Counter

transactions = ["electronics", "food", "clothing", "food", "food", "electronics"]

counts = Counter(transactions)
print(counts)                   # Counter({'food': 3, 'electronics': 2, 'clothing': 1})
print(counts.most_common(2))    # [('food', 3), ('electronics', 2)]

# Арифметика Counter
a = Counter(["x", "y", "x"])
b = Counter(["y", "y", "z"])
print(a + b)   # Counter({'y': 3, 'x': 2, 'z': 1})
print(a - b)   # Counter({'x': 2}) — только положительные значения

In [ ]:
from collections import defaultdict

orders = [
    {"category": "electronics", "amount": 500},
    {"category": "food",        "amount": 30},
    {"category": "electronics", "amount": 200},
    {"category": "food",        "amount": 45},
]

# ❌ Ручной подход
totals_manual = {}
for order in orders:
    cat = order["category"]
    if cat not in totals_manual:
        totals_manual[cat] = 0
    totals_manual[cat] += order["amount"]

# ✅ defaultdict
totals = defaultdict(float)
for order in orders:
    totals[order["category"]] += order["amount"]

print(dict(totals))  # {'electronics': 700.0, 'food': 75.0}

# defaultdict(list) — группировка
by_category = defaultdict(list)
for order in orders:
    by_category[order["category"]].append(order["amount"])

print(dict(by_category))
# {'electronics': [500, 200], 'food': [30, 45]}

#### namedtuple — читаемые записи без полноценного класса

In [ ]:
from collections import namedtuple

Transaction = namedtuple("Transaction", ["id", "category", "amount", "currency"])

t = Transaction(id=1, category="electronics", amount=500.0, currency="USD")
print(t.amount)       # 500.0 — доступ по атрибуту
print(t[2])           # 500.0 — также доступ по индексу
print(t._asdict())    # OrderedDict для удобной сериализации в JSON

#### Синтаксис генераторов

In [ ]:
# Генератор списка
squares = [x**2 for x in range(10) if x % 2 == 0]
# [0, 4, 16, 36, 64]

# Генератор словаря
prices = {"apple": 1.2, "banana": 0.5, "cherry": 3.0}
expensive = {k: v for k, v in prices.items() if v > 1.0}
# {'apple': 1.2, 'cherry': 3.0}

# Генератор множества
categories = {"food", "food", "tech", "food", "tech"}  # ← дедупликация при создании

# Выражение-генератор (ленивое — список в памяти не создаётся)
total = sum(x**2 for x in range(1_000_000))  # эффективно по памяти

In [ ]:
# Типичный ответ API — список словарей с вложенными полями
orders = [
    {"id": 1, "customer": {"country": "DE"}, "items": [{"sku": "A1", "qty": 2}, {"sku": "B2", "qty": 1}]},
    {"id": 2, "customer": {"country": "FR"}, "items": [{"sku": "A1", "qty": 1}]},
    {"id": 3, "customer": {"country": "DE"}, "items": [{"sku": "C3", "qty": 5}]},
]

# 1. Распрямляем все позиции
all_items = [item for order in orders for item in order["items"]]
print(all_items)
# [{'sku': 'A1', 'qty': 2}, {'sku': 'B2', 'qty': 1}, {'sku': 'A1', 'qty': 1}, {'sku': 'C3', 'qty': 5}]

# 2. Суммарное количество по SKU
qty_per_sku = Counter()
for item in all_items:
    qty_per_sku[item["sku"]] += item["qty"]
print(dict(qty_per_sku))  # {'A1': 3, 'B2': 1, 'C3': 5}

# 3. Заказы по странам
orders_by_country = defaultdict(list)
for order in orders:
    country = order.get("customer", {}).get("country", "UNKNOWN")
    orders_by_country[country].append(order["id"])
print(dict(orders_by_country))  # {'DE': [1, 3], 'FR': [2]}